# Continuous K-Factor

Fit K-factor `LogisticFM` models on continuous bounded response matrices, using the `torch_measure` Beta likelihood rather than Bernoulli loss.


In [ ]:
!python -m pip install -e ..


In [ ]:
import json
from functools import partial
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from torch_measure.data import ResponseMatrix
from torch_measure.fitting._losses import beta_nll
from torch_measure.models import LogisticFM, predict_dense
from torch_measure.models.rotation import varimax_rotation

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


## Load Continuous Response Matrix

Change `KFACTOR_MATRIX` to switch between continuous response matrices. Values of `-1` and `NaN` are treated as missing. Exact 0/1 scores are clipped only for the Beta likelihood.


In [ ]:
MATRIX_PRESETS = {
    "harmmetric_eval": {
        "prefix": "harmmetric_eval",
        "dir": "../benchmarks/HarmMetric_Eval/response_matrices",
        "benchmark_id": "harmmetric_eval",
        "item_content_field": "source",
        "item_id_field": "prompt_id",
        "value": "overall_effectiveness_score: soft/fractional score in [0, 1]",
    },
}

KFACTOR_MATRIX = "harmmetric_eval"
config = MATRIX_PRESETS[KFACTOR_MATRIX]
matrix_dir = config["dir"]
prefix = config["prefix"]

matrix_path = f"{matrix_dir}/{prefix}_response_matrix.csv"
subject_meta_path = f"{matrix_dir}/{prefix}_subject_metadata.csv"
item_meta_path = f"{matrix_dir}/{prefix}_item_metadata.csv"

print(f"Loading {KFACTOR_MATRIX}")
print(matrix_path)

df = pd.read_csv(matrix_path, index_col=0)
subject_meta = pd.read_csv(subject_meta_path)
item_meta = pd.read_csv(item_meta_path)

item_content_field = config.get("item_content_field")
if item_content_field in item_meta.columns:
    item_contents = list(item_meta[item_content_field].fillna("").astype(str))
else:
    item_contents = list(item_meta.iloc[:, 0].astype(str))

rm = ResponseMatrix(
    data=torch.tensor(df.values, dtype=torch.float32),
    subject_ids=list(df.index.astype(str)),
    item_ids=list(df.columns.astype(str)),
    item_contents=item_contents,
    subject_metadata=subject_meta.to_dict("records"),
    info={
        "benchmark_id": config["benchmark_id"],
        "item_id_field": config["item_id_field"],
        "value": config["value"],
    },
)

print(f"{rm.n_subjects} models x {rm.n_items} items, density = {rm.density:.1%}")
display(df.head())


## Fit K-Factor Models

The model is still `sigmoid(U @ V.T + Z)`, but fitting uses `torch_measure.fitting._losses.beta_nll` with fixed precision `phi`.


In [ ]:
BETA_PHI = 10.0
BETA_EPS = 1e-4

# Missing values are NaN or -1. Fit values are clipped into the open interval required by beta_nll.
data = rm.data.to(device).float()
mask = ~torch.isnan(data) & (data != -1)
fit_data = data.clamp(BETA_EPS, 1 - BETA_EPS)

print(f"Continuous K-factor data: {data.shape[0]} models x {data.shape[1]} items, observed={mask.float().mean().item():.1%}")
print(f"Beta likelihood phi={BETA_PHI}, clip eps={BETA_EPS}")


In [ ]:
def continuous_metrics(predicted, observed, mask, phi=BETA_PHI, eps=BETA_EPS):
    p = predicted[mask].clamp(eps, 1 - eps)
    y = observed[mask].float().clamp(eps, 1 - eps)
    residual = p - y

    if len(y) > 1:
        pearson = torch.corrcoef(torch.stack([p.detach().cpu(), y.detach().cpu()]))[0, 1].item()
    else:
        pearson = float("nan")

    return {
        "beta_nll": beta_nll(p, y, phi=phi).item(),
        "mse": residual.pow(2).mean().item(),
        "rmse": torch.sqrt(residual.pow(2).mean()).item(),
        "mae": residual.abs().mean().item(),
        "pearson": pearson,
    }


def fit_logistic_fm_continuous_k(data, fit_data, mask, k, device="cpu", max_epochs=1000, lr=0.03, phi=BETA_PHI, seed=123):
    torch.manual_seed(seed + k)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed + k)

    model = LogisticFM(
        n_subjects=data.shape[0],
        n_items=data.shape[1],
        n_factors=k,
        device=device,
    )

    history = model.fit(
        fit_data,
        mask=mask,
        method="mle",
        max_epochs=max_epochs,
        lr=lr,
        verbose=True,
        loss_fn=partial(beta_nll, phi=phi),
    )

    with torch.no_grad():
        probs = predict_dense(model).detach()
    metrics = continuous_metrics(probs, data, mask, phi=phi)

    return {"model": model, "history": history, "metrics": metrics}


kfactor_specs = {
    1: {"max_epochs": 1000, "lr": 0.03},
    2: {"max_epochs": 1000, "lr": 0.02},
}

kfactor_results = {}
for k, spec in kfactor_specs.items():
    print(f"\nFitting continuous LogisticFM K={k}")
    kfactor_results[k] = fit_logistic_fm_continuous_k(
        data,
        fit_data,
        mask,
        k=k,
        device=device,
        max_epochs=spec["max_epochs"],
        lr=spec["lr"],
        phi=BETA_PHI,
        seed=123,
    )

kfactor_fit_summary = pd.DataFrame([
    {
        "k": k,
        "loss": result["history"]["losses"][-1],
        **result["metrics"],
    }
    for k, result in kfactor_results.items()
])

display(kfactor_fit_summary.round(4))


## Item Difficulties With Laplace Uncertainty

For `LogisticFM`, `Z` is item easiness and `difficulty = -Z`. The SE below uses a diagonal/conditional Laplace approximation under the fixed-precision Beta likelihood.


In [ ]:
def build_item_difficulty_table(result, rm, item_meta=None):
    model = result["model"]

    difficulty = model.difficulty.detach().cpu()
    difficulty_centered = difficulty - difficulty.mean()
    easiness_z = model.Z.detach().cpu()
    loadings = model.loadings.detach().cpu()

    rotated_loadings, rotation = varimax_rotation(loadings)
    rotated_abilities = model.U.detach().cpu() @ rotation

    with torch.no_grad():
        predicted = predict_dense(model).detach().cpu()
        data_cpu = data.detach().cpu()
        mask_cpu = mask.detach().cpu()
        observed_mean = torch.where(mask_cpu, data_cpu, torch.nan).nanmean(dim=0)
        predicted_mean = torch.where(mask_cpu, predicted, torch.nan).nanmean(dim=0)
        n_observed = mask_cpu.sum(dim=0)

    table = pd.DataFrame({
        "item_id": rm.item_ids,
        "difficulty": difficulty.numpy(),
        "difficulty_centered": difficulty_centered.numpy(),
        "easiness_z": easiness_z.numpy(),
        "observed_mean_score": observed_mean.numpy(),
        "predicted_mean_score": predicted_mean.numpy(),
        "n_observed": n_observed.numpy(),
    })

    for j in range(rotated_loadings.shape[1]):
        table[f"loading_factor_{j + 1}"] = rotated_loadings[:, j].numpy()

    loading_cols = [c for c in table.columns if c.startswith("loading_factor_")]
    table["dominant_factor"] = (
        table[loading_cols]
        .abs()
        .idxmax(axis=1)
        .str.replace("loading_factor_", "factor_", regex=False)
    )

    if item_meta is not None:
        table = table.merge(
            item_meta.reset_index(drop=True),
            left_index=True,
            right_index=True,
            how="left",
            suffixes=("", "_meta"),
        )

    model_factors = pd.DataFrame({"model": rm.subject_ids})
    for j in range(rotated_abilities.shape[1]):
        model_factors[f"factor_{j + 1}"] = rotated_abilities[:, j].numpy()

    return table, model_factors


item_difficulty_tables = {}
model_factor_tables = {}
for k, result in kfactor_results.items():
    item_table, model_table = build_item_difficulty_table(result, rm, item_meta=item_meta)
    item_difficulty_tables[k] = item_table
    model_factor_tables[k] = model_table

print("K=1 hardest/lowest-score items")
display(item_difficulty_tables[1].sort_values("difficulty_centered", ascending=False).head(20).round(3))

print("K=2 hardest/lowest-score items")
display(item_difficulty_tables[2].sort_values("difficulty_centered", ascending=False).head(20).round(3))


In [ ]:
def item_difficulty_laplace_se_beta(model, data, mask, phi=BETA_PHI, eps=BETA_EPS):
    # Conditional diagonal Laplace SE for LogisticFM item difficulty under Beta NLL.
    data = data.to(model.device).float()
    mask = mask.to(model.device)

    with torch.no_grad():
        mu = predict_dense(model).clamp(eps, 1 - eps)
        alpha = mu * phi
        beta = (1 - mu) * phi

        # Expected Fisher information for eta where mu=sigmoid(eta):
        # I_eta = phi^2 * (trigamma(alpha) + trigamma(beta)) * (dmu/deta)^2.
        dmu_deta = mu * (1 - mu)
        info_matrix = (phi ** 2) * (torch.polygamma(1, alpha) + torch.polygamma(1, beta)) * dmu_deta.pow(2)
        info = torch.where(mask, info_matrix, torch.zeros_like(info_matrix)).sum(dim=0)
        raw_se = 1 / torch.sqrt(info.clamp_min(1e-8))

        n_items = raw_se.numel()
        raw_var = raw_se.pow(2)
        centered_var = ((1 - 1 / n_items) ** 2) * raw_var + (raw_var.sum() - raw_var) / (n_items ** 2)
        centered_se = torch.sqrt(centered_var.clamp_min(1e-12))

    return raw_se.detach().cpu(), centered_se.detach().cpu()


item_difficulty_with_uncertainty = {}
for k, result in kfactor_results.items():
    raw_se, centered_se = item_difficulty_laplace_se_beta(result["model"], data, mask, phi=BETA_PHI)

    table = item_difficulty_tables[k].copy()
    table["difficulty_laplace_se"] = raw_se.numpy()
    table["difficulty_centered_laplace_se"] = centered_se.numpy()
    table["difficulty_centered_laplace_lo"] = table["difficulty_centered"] - 1.96 * table["difficulty_centered_laplace_se"]
    table["difficulty_centered_laplace_hi"] = table["difficulty_centered"] + 1.96 * table["difficulty_centered_laplace_se"]
    item_difficulty_with_uncertainty[k] = table.sort_values("difficulty_centered", ascending=False).reset_index(drop=True)

print("K=1 item difficulties with Laplace uncertainty")
display(item_difficulty_with_uncertainty[1].round(3))

print("K=2 item difficulties with Laplace uncertainty")
display(item_difficulty_with_uncertainty[2].round(3))


## Save Results


In [ ]:
result_name = KFACTOR_MATRIX
out_dir = Path("results_continuous") / result_name
out_dir.mkdir(parents=True, exist_ok=True)

def item_score_records(df, item_ids):
    # Return {item_id: {model: observed score}} with NaN/-1 converted to None.
    score_map = {}
    for item_id in item_ids:
        values = df[str(item_id)] if str(item_id) in df.columns else df[item_id]
        score_map[str(item_id)] = {
            str(model): (None if pd.isna(value) or float(value) == -1.0 else float(value))
            for model, value in values.items()
        }
    return score_map

scores_by_item = item_score_records(df, rm.item_ids)

kfactor_fit_summary.to_csv(out_dir / f"{result_name}_continuous_kfactor_fit_summary.csv", index=False)
item_difficulty_json = {
    "matrix": result_name,
    "benchmark_id": rm.info.get("benchmark_id"),
    "item_id_field": rm.info.get("item_id_field"),
    "value": rm.info.get("value"),
    "likelihood": "Beta likelihood with LogisticFM mean, fixed phi",
    "beta_phi": BETA_PHI,
    "beta_eps": BETA_EPS,
    "fits": {},
}

for k, table in item_difficulty_with_uncertainty.items():
    table.to_csv(out_dir / f"{result_name}_continuous_kfactor_k{k}_item_difficulties_with_laplace_uncertainty.csv", index=False)
    model_factor_tables[k].to_csv(out_dir / f"{result_name}_continuous_kfactor_k{k}_model_factors.csv", index=False)

    records = table.astype(object).where(pd.notna(table), None).to_dict("records")
    for record in records:
        record["scores"] = scores_by_item.get(str(record["item_id"]), {})
    item_difficulty_json["fits"][f"k{k}"] = records

with open(out_dir / f"{result_name}_continuous_kfactor_item_difficulties_with_laplace_uncertainty.json", "w") as f:
    json.dump(item_difficulty_json, f, indent=2)

print(f"Saved continuous K-factor tables to {out_dir.resolve()}")
